In [ ]:
%run "./02_EMS_preprocessing.ipynb"

import numpy as np
import pandas as pd

files = {
    "train": PROCESSED_DIR / "train_with_symbolic_states.csv",
    "validation": PROCESSED_DIR / "validation_with_symbolic_states.csv",
    "test": PROCESSED_DIR / "test_with_symbolic_states.csv",
}

for path in files.values():
    if not path.exists():
        raise FileNotFoundError(f"Fichier absent : {path}")

train_df = pd.read_csv(files["train"])
val_df = pd.read_csv(files["validation"])
test_df = pd.read_csv(files["test"])

datasets = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

print(train_df.shape, val_df.shape, test_df.shape)

ROOT_DIR  : C:\Users\Admin\Desktop\Projet_Artemis2
DATA_FILE : C:\Users\Admin\Desktop\Projet_Artemis2\data\Artemis.csv | existe: True
RANDOM_SEED: 42 | DEVICE: cuda
CONFIGURATION DU PROJET HESS

BATTERIE ÉNERGIE
Énergie          : 13709.89 Wh
Tension          : 450.00 V
Capacité         : 30.4664 Ah
Masse            : 55.12 kg
Courant recharge : -14.00 A
Courant décharge : 28.00 A
Configuration    : 125S7P

BATTERIE PUISSANCE
Énergie          : 2987.12 Wh
Tension          : 402.60 V
Capacité         : 7.4196 Ah
Masse            : 50.02 kg
Courant recharge : -130.00 A
Courant décharge : 400.00 A
Configuration    : 122S2P

HESS
Énergie totale   : 16697.01 Wh
Masse totale     : 105.14 kg
Tension du bus   : 402.60 V
Puissance min    : -58638.00 W
Puissance max    : 173640.00 W

MODÈLES
LSTM seul        : 7 → 64 → 3
LSTM NS          : 15 → 64 → 3
GNN simple       : 12 → 32 → 1
MLP simple       : 5 → 64 → 32 → 1
MLP NS           : 17 → 64 → 32 → 1

Device           : cuda
Fichier          : 

Fonctions d’appartenance

In [22]:
def left_shoulder(x, full_until, zero_from):
    if full_until >= zero_from:
        raise ValueError("Il faut full_until < zero_from.")

    x = np.asarray(x, dtype=float)

    return np.where(
        x <= full_until,
        1.0,
        np.where(
            x >= zero_from,
            0.0,
            (zero_from - x)
            / (zero_from - full_until),
        ),
    )


def right_shoulder(x, zero_until, full_from):
    if zero_until >= full_from:
        raise ValueError("Il faut zero_until < full_from.")

    x = np.asarray(x, dtype=float)

    return np.where(
        x <= zero_until,
        0.0,
        np.where(
            x >= full_from,
            1.0,
            (x - zero_until)
            / (full_from - zero_until),
        ),
    )


def trapmf(x, a, b, c, d):
    if not a < b <= c < d:
        raise ValueError(
            "Il faut respecter a < b <= c < d."
        )

    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x, dtype=float)

    left = (x > a) & (x < b)
    plateau = (x >= b) & (x <= c)
    right = (x > c) & (x < d)

    y[left] = (x[left] - a) / (b - a)
    y[plateau] = 1.0
    y[right] = (d - x[right]) / (d - c)

    return y

Appartenances SOC, puissance et accélération


In [23]:
SOC_LOW_FULL_EB = (
    SOC_EB_MIN + SOC_LOW_THRESHOLD
) / 2.0

SOC_LOW_FULL_PB = (
    SOC_PB_MIN + SOC_LOW_THRESHOLD
) / 2.0


def soc_eb_low(value):
    return left_shoulder(
        value,
        SOC_LOW_FULL_EB,
        SOC_LOW_THRESHOLD,
    )


def soc_pb_low(value):
    return left_shoulder(
        value,
        SOC_LOW_FULL_PB,
        SOC_LOW_THRESHOLD,
    )


def soc_medium(value):
    return trapmf(
        value,
        0.25,
        0.35,
        0.65,
        0.75,
    )


def soc_high(value):
    return right_shoulder(
        value,
        0.70,
        0.80,
    )


def power_strong_recharge(value):
    return left_shoulder(
        value,
        P_EB_MIN_W - 2000.0,
        P_EB_MIN_W,
    )


def power_moderate_recharge(value):
    return trapmf(
        value,
        P_EB_MIN_W - 1000.0,
        P_EB_MIN_W,
        -2.0 * EPS_POWER_W,
        -EPS_POWER_W,
    )


def power_zero(value):
    return trapmf(
        value,
        -2.0 * EPS_POWER_W,
        -EPS_POWER_W,
        EPS_POWER_W,
        2.0 * EPS_POWER_W,
    )


def power_moderate_traction(value):
    return trapmf(
        value,
        EPS_POWER_W,
        2.0 * EPS_POWER_W,
        0.60 * P_EB_MAX_W,
        P_EB_MAX_W,
    )


def power_strong_traction(value):
    return right_shoulder(
        value,
        0.60 * P_EB_MAX_W,
        P_EB_MAX_W,
    )


def acceleration_braking(value):
    return left_shoulder(
        value,
        -1.20,
        -0.20,
    )


def acceleration_stable(value):
    return trapmf(
        value,
        -0.40,
        -0.10,
        0.10,
        0.40,
    )

Base de règles floues

In [24]:
FUZZY_DEFAULT_ALPHA = 0.30

FUZZY_RULE_NAMES = [
    "R1_PB_low_traction",
    "R2_EB_low_PB_available",
    "R3_strong_traction",
    "R4_zero_demand",
    "R5_regenerative_braking",
    "R5b_PB_high_recharge",
    "R7_two_low_SOC",
]

FUZZY_RULE_CONSEQUENTS = np.array(
    [
        0.20,
        0.75,
        0.75,
        0.20,
        0.75,
        0.20,
        0.50,
    ],
    dtype=float,
)

Calcul de `alpha_fuzzy`

In [25]:

def alpha_fuzzy_calc(
    soc_eb,
    soc_pb,
    p_dem,
    acceleration,
):
    soc_eb = np.asarray(soc_eb, dtype=float)
    soc_pb = np.asarray(soc_pb, dtype=float)
    p_dem = np.asarray(p_dem, dtype=float)
    acceleration = np.asarray(
        acceleration,
        dtype=float,
    )

    eb_low = soc_eb_low(soc_eb)
    pb_low = soc_pb_low(soc_pb)
    pb_medium = soc_medium(soc_pb)
    pb_high = soc_high(soc_pb)

    strong_recharge = power_strong_recharge(
        p_dem
    )

    moderate_recharge = (
        power_moderate_recharge(p_dem)
    )

    zero_demand = power_zero(p_dem)

    moderate_traction = (
        power_moderate_traction(p_dem)
    )

    strong_traction = (
        power_strong_traction(p_dem)
    )

    stable = acceleration_stable(
        acceleration
    )

    traction = np.maximum(
        moderate_traction,
        strong_traction,
    )

    recharge_power = np.maximum(
        strong_recharge,
        moderate_recharge,
    )

    recharge = np.where(
        p_dem < -EPS_POWER_W,
        recharge_power,
        0.0,
    )

    pb_available = np.maximum(
        pb_medium,
        pb_high,
    )

    pb_rechargeable = np.maximum(
        pb_low,
        pb_medium,
    )

    w1 = np.minimum(
        pb_low,
        traction,
    )

    w2 = np.minimum.reduce([
        eb_low,
        pb_available,
        traction,
    ])

    w3 = np.minimum(
        strong_traction,
        pb_available,
    )

    w4 = np.minimum(
        zero_demand,
        stable,
    )

    w5 = np.minimum(
        recharge,
        pb_rechargeable,
    )

    w5b = np.minimum(
        recharge,
        pb_high,
    )

    w7 = np.minimum(
        eb_low,
        pb_low,
    )

    strengths = np.stack(
        [
            w1,
            w2,
            w3,
            w4,
            w5,
            w5b,
            w7,
        ],
        axis=-1,
    )

    strength_sum = strengths.sum(
        axis=-1
    )

    coverage = strength_sum > 1e-9

    weighted_sum = (
        strengths
        * FUZZY_RULE_CONSEQUENTS
    ).sum(axis=-1)

    alpha = np.full(
        strength_sum.shape,
        FUZZY_DEFAULT_ALPHA,
        dtype=float,
    )

    np.divide(
        weighted_sum,
        strength_sum,
        out=alpha,
        where=coverage,
    )

    alpha = np.clip(
        alpha,
        0.0,
        1.0,
    )

    dominant_index = np.argmax(
        strengths,
        axis=-1,
    )

    rule_names = np.asarray(
        FUZZY_RULE_NAMES,
        dtype=object,
    )

    dominant_rule = np.where(
        coverage,
        rule_names[dominant_index],
        "DEFAULT",
    )

    return {
        "alpha": alpha,
        "strengths": strengths,
        "coverage": coverage,
        "dominant_rule": dominant_rule,
    }



Stratégie EB-priority exacte

In [26]:
def eb_priority_alpha(
    p_dem,
    soc_eb,
    p_eb_min=P_EB_MIN_W,
    p_eb_max=P_EB_MAX_W,
    soc_min=SOC_EB_MIN,
    eps=EPS_POWER_W,
):
    p_dem = np.asarray(
        p_dem,
        dtype=float,
    )

    soc_eb = np.asarray(
        soc_eb,
        dtype=float,
    )

    protected = soc_eb <= soc_min

    p_eb = np.zeros_like(
        p_dem,
        dtype=float,
    )

    p_eb = np.where(
        p_dem < p_eb_min,
        p_eb_min,
        p_eb,
    )

    p_eb = np.where(
        (p_dem >= p_eb_min)
        & (p_dem < 0.0),
        p_dem,
        p_eb,
    )

    p_eb = np.where(
        (p_dem >= 0.0)
        & protected,
        0.0,
        p_eb,
    )

    p_eb = np.where(
        (p_dem > 0.0)
        & (p_dem <= p_eb_max)
        & ~protected,
        p_dem,
        p_eb,
    )

    p_eb = np.where(
        (p_dem > p_eb_max)
        & ~protected,
        p_eb_max,
        p_eb,
    )

    p_pb = p_dem - p_eb

    valid = np.abs(p_dem) > eps

    alpha = np.full_like(
        p_dem,
        np.nan,
        dtype=float,
    )

    np.divide(
        p_pb,
        p_dem,
        out=alpha,
        where=valid,
    )

    invalid_alpha = (
        valid
        & (
            (alpha < -1e-9)
            | (alpha > 1.0 + 1e-9)
        )
    )

    if invalid_alpha.any():
        raise ValueError(
            "Alpha EB-priority hors de [0,1]."
        )

    alpha[valid] = np.clip(
        alpha[valid],
        0.0,
        1.0,
    )

    return alpha

Tests des scénarios


In [27]:
scenarios = {
    "demande nulle":
        (0.60, 0.60, 0.0, 0.0),

    "forte traction":
        (0.85, 0.85, 30000.0, 2.5),

    "forte recharge":
        (0.60, 0.60, -20000.0, -3.0),

    "PB faible":
        (0.85, 0.22, 5000.0, 0.1),

    "EB faible":
        (0.22, 0.85, 5000.0, 0.1),

    "deux SOC faibles":
        (0.22, 0.25, 8000.0, 0.3),

    "PB presque pleine pendant recharge":
        (0.50, 0.97, -8000.0, -1.5),
}

scenario_rows = []

for name, values in scenarios.items():
    soc_eb, soc_pb, p_dem, acceleration = values

    result = alpha_fuzzy_calc(
        np.array([soc_eb]),
        np.array([soc_pb]),
        np.array([p_dem]),
        np.array([acceleration]),
    )

    alpha_fuzzy = float(
        result["alpha"][0]
    )

    alpha_priority = eb_priority_alpha(
        np.array([p_dem]),
        np.array([soc_eb]),
    )[0]

    scenario_rows.append({
        "scenario": name,
        "SOC_EB": soc_eb,
        "SOC_PB": soc_pb,
        "Pdem": p_dem,
        "acceleration": acceleration,
        "alpha_fuzzy": alpha_fuzzy,
        "alpha_eb_priority": alpha_priority,
        "dominant_rule": (
            result["dominant_rule"][0]
        ),
        "rule_coverage": bool(
            result["coverage"][0]
        ),
    })

scenario_df = pd.DataFrame(
    scenario_rows
)

display(scenario_df)

,scenario,SOC_EB,SOC_PB,Pdem,acceleration,alpha_fuzzy,alpha_eb_priority,dominant_rule,rule_coverage
0,demande nulle,0.60,0.60,0.0,0.0,0.200000,NaN,R4_zero_demand,True
1,forte traction,0.85,0.85,30000.0,2.5,0.750000,0.5800,R3_strong_traction,True
2,forte recharge,0.60,0.60,-20000.0,-3.0,0.750000,0.6850,R5_regenerative_braking,True
3,PB faible,0.85,0.22,5000.0,0.1,0.200000,0.0000,R1_PB_low_traction,True
4,EB faible,0.22,0.85,5000.0,0.1,0.750000,0.0000,R2_EB_low_PB_available,True
5,deux SOC faibles,0.22,0.25,8000.0,0.3,0.356846,0.0000,R7_two_low_SOC,True
6,PB presque pleine pendant recharge,0.50,0.97,-8000.0,-1.5,0.200000,0.2125,R5b_PB_high_recharge,True


Application aux trois splits

In [28]:
def apply_fuzzy_controller(data):
    result = alpha_fuzzy_calc(
        data["SOC_EB"].to_numpy(),
        data["SOC_PB"].to_numpy(),
        data["hasPower"].to_numpy(),
        data["hasAcceleration"].to_numpy(),
    )

    output = data.copy()

    output["alpha_ems_fuzzy_logic"] = result["alpha"]

    output["fuzzy_rule_covered"] = (
        result["coverage"].astype(np.int8)
    )

    output["fuzzy_default_used"] = (
        ~result["coverage"]
    ).astype(np.int8)

    output["fuzzy_dominant_rule"] = (
        result["dominant_rule"]
    )

    for index, rule_name in enumerate(
        FUZZY_RULE_NAMES
    ):
        output[
            f"strength_{rule_name}"
        ] = result["strengths"][:, index]

    output["P_PB_ems_fuzzy_logic"] = (
        output["alpha_ems_fuzzy_logic"]
        * output["hasPower"]
    )

    output["P_EB_ems_fuzzy_logic"] = (
        1.0 - output["alpha_ems_fuzzy_logic"]
    ) * output["hasPower"]

    output["power_balance_error_fuzzy"] = (
        output["hasPower"]
        - output["P_EB_ems_fuzzy_logic"]
        - output["P_PB_ems_fuzzy_logic"]
    )

    output["P_EB_fuzzy_violation"] = (
        (
            output["P_EB_ems_fuzzy_logic"]
            < P_EB_MIN_W
        )
        | (
            output["P_EB_ems_fuzzy_logic"]
            > P_EB_MAX_W
        )
    ).astype(np.int8)

    output["P_PB_fuzzy_violation"] = (
        (
            output["P_PB_ems_fuzzy_logic"]
            < P_PB_MIN_W
        )
        | (
            output["P_PB_ems_fuzzy_logic"]
            > P_PB_MAX_W
        )
    ).astype(np.int8)

    output[
        "EB_protection_fuzzy_violation"
    ] = (
        (
            output["SOC_EB"]
            <= SOC_EB_MIN
        )
        & (
            output["P_EB_ems_fuzzy_logic"]
            > EPS_POWER_W
        )
    ).astype(np.int8)

    output[
        "PB_protection_fuzzy_violation"
    ] = (
        (
            output["SOC_PB"]
            <= SOC_PB_MIN
        )
        & (
            output["P_PB_ems_fuzzy_logic"]
            > EPS_POWER_W
        )
    ).astype(np.int8)

    return output

Exécution et sauvegarde

In [29]:
fuzzy_datasets = {}

for split_name, data in datasets.items():
    fuzzy_data = apply_fuzzy_controller(
        data
    )

    fuzzy_datasets[split_name] = (
        fuzzy_data
    )

    output_file = (
        PROCESSED_DIR
        / f"{split_name}_fuzzy_results.csv"
    )

    fuzzy_data.to_csv(
        output_file,
        index=False,
    )

    print(
        split_name,
        "| alpha moyen :",
        fuzzy_data["alpha_ems_fuzzy_logic"].mean(),
        "| couverture :",
        100.0
        * fuzzy_data[
            "fuzzy_rule_covered"
        ].mean(),
        "%",
        "| fichier :",
        output_file,
    )

train | alpha moyen : 0.36600882849277355 | couverture : 72.34686854783206 % | fichier : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\train_fuzzy_results.csv
validation | alpha moyen : 0.47269436903121975 | couverture : 71.61343612334802 % | fichier : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\validation_fuzzy_results.csv
test | alpha moyen : 0.6013316018996051 | couverture : 89.23754472887421 % | fichier : C:\Users\Admin\Desktop\Projet_Artemis2\data\processed\test_fuzzy_results.csv


Diagnostic de couverture et de faisabilité


In [30]:
diagnostic_rows = []

for split_name, data in fuzzy_datasets.items():
    diagnostic_rows.append({
        "split": split_name,
        "rule_coverage_percent": (
            100.0
            * data[
                "fuzzy_rule_covered"
            ].mean()
        ),
        "default_percent": (
            100.0
            * data[
                "fuzzy_default_used"
            ].mean()
        ),
        "P_EB_violations": int(
            data[
                "P_EB_fuzzy_violation"
            ].sum()
        ),
        "P_PB_violations": int(
            data[
                "P_PB_fuzzy_violation"
            ].sum()
        ),
        "EB_protection_violations": int(
            data[
                "EB_protection_fuzzy_violation"
            ].sum()
        ),
        "PB_protection_violations": int(
            data[
                "PB_protection_fuzzy_violation"
            ].sum()
        ),
        "max_balance_error_W": float(
            data[
                "power_balance_error_fuzzy"
            ].abs().max()
        ),
    })

fuzzy_diagnostic_df = pd.DataFrame(
    diagnostic_rows
)

display(fuzzy_diagnostic_df)

fuzzy_diagnostic_df.to_csv(
    TABLES_DIR
    / "fuzzy_controller_diagnostics.csv",
    index=False,
)

,split,rule_coverage_percent,default_percent,P_EB_violations,P_PB_violations,EB_protection_violations,PB_protection_violations,max_balance_error_W
0,train,72.346869,27.653131,617,0,0,0,5.456968e-12
1,validation,71.613436,28.386564,168,0,0,0,3.637979e-12
2,test,89.237545,10.762455,64,0,369,0,3.637979e-12


Comparaison globale sur le test

In [31]:
test_result = fuzzy_datasets["test"]

alpha_priority = eb_priority_alpha(
    test_result["hasPower"].to_numpy(),
    test_result["SOC_EB"].to_numpy(),
)

alpha_fuzzy = test_result[
    "alpha_ems_fuzzy_logic"
].to_numpy()

valid = (
    np.abs(
        test_result["hasPower"].to_numpy()
    ) > EPS_POWER_W
) & np.isfinite(alpha_priority)

alpha_priority_valid = (
    alpha_priority[valid]
)

alpha_fuzzy_valid = (
    alpha_fuzzy[valid]
)

if (
    np.std(alpha_priority_valid) > 0
    and np.std(alpha_fuzzy_valid) > 0
):
    correlation = float(
        np.corrcoef(
            alpha_fuzzy_valid,
            alpha_priority_valid,
        )[0, 1]
    )
else:
    correlation = np.nan

mae_alpha = float(
    np.mean(
        np.abs(
            alpha_fuzzy_valid
            - alpha_priority_valid
        )
    )
)

comparison_summary = pd.DataFrame([
    {
        "correlation": correlation,
        "mae_alpha": mae_alpha,
        "mean_alpha_fuzzy": float(
            alpha_fuzzy_valid.mean()
        ),
        "mean_alpha_eb_priority": float(
            alpha_priority_valid.mean()
        ),
        "valid_samples": int(valid.sum()),
    }
])

display(comparison_summary)

comparison_summary.to_csv(
    TABLES_DIR
    / "fuzzy_vs_eb_priority_global.csv",
    index=False,
)

,correlation,mae_alpha,mean_alpha_fuzzy,mean_alpha_eb_priority,valid_samples
0,0.068736,0.539376,0.674298,0.222509,3066


Analyse par régime

In [32]:
regimes = {
    "traction": (
        test_result["hasPower"]
        > EPS_POWER_W
    ),
    "regeneration": (
        test_result["hasPower"]
        < -EPS_POWER_W
    ),
    "high_power": (
        test_result["hasPower"].abs()
        >= HIGH_POWER_THRESHOLD_W
    ),
    "EB_low_SOC": (
        test_result["EB_low_SOC"] == 1
    ),
    "PB_low_SOC": (
        test_result["PB_low_SOC"] == 1
    ),
    "normal_SOC": (
        (test_result["EB_low_SOC"] == 0)
        & (test_result["PB_low_SOC"] == 0)
    ),
}

regime_rows = []

for regime_name, regime_mask in regimes.items():
    mask = (
        regime_mask.to_numpy()
        & valid
    )

    if mask.sum() == 0:
        continue

    fuzzy_values = alpha_fuzzy[mask]
    priority_values = alpha_priority[mask]

    regime_rows.append({
        "regime": regime_name,
        "samples": int(mask.sum()),
        "alpha_fuzzy_mean": float(
            fuzzy_values.mean()
        ),
        "alpha_priority_mean": float(
            priority_values.mean()
        ),
        "mae_alpha": float(
            np.mean(
                np.abs(
                    fuzzy_values
                    - priority_values
                )
            )
        ),
        "mean_difference": float(
            np.mean(
                fuzzy_values
                - priority_values
            )
        ),
    })

regime_df = pd.DataFrame(
    regime_rows
)

display(regime_df)

regime_df.to_csv(
    TABLES_DIR
    / "fuzzy_vs_eb_priority_by_regime.csv",
    index=False,
)

,regime,samples,alpha_fuzzy_mean,alpha_priority_mean,mae_alpha,mean_difference
0,traction,2042,0.642233,0.250387,0.519392,0.391846
1,regeneration,1024,0.738241,0.166916,0.579226,0.571325
2,high_power,319,0.726216,0.691414,0.190293,0.034802
3,EB_low_SOC,2081,0.721026,0.285150,0.564465,0.435876
4,PB_low_SOC,199,0.474794,0.641048,0.493898,-0.166253
5,normal_SOC,985,0.575575,0.090167,0.486369,0.485408


Vérification de l’équivalence avec `alpha_ems_eb_priority`


In [33]:

if "alpha_ems_eb_priority" in test_result.columns:
    alpha_ems_eb_priority = test_result[
        "alpha_ems_eb_priority"
    ].to_numpy(dtype=float)

    comparison_valid = (
        valid
        & np.isfinite(alpha_ems_eb_priority)
    )

    max_difference = float(
        np.max(
            np.abs(
                alpha_priority[
                    comparison_valid
                ]
                - alpha_ems_eb_priority[
                    comparison_valid
                ]
            )
        )
    )

    print(
        "Écart maximal alpha_ems_eb_priority / EB-priority :",
        max_difference,
    )

    if max_difference > 1e-8:
        raise ValueError(
            "alpha_ems_eb_priority et EB-priority "
            "ne sont pas équivalents."
        )



Écart maximal alpha_ems_eb_priority / EB-priority : 1.6653345369377348e-16


In [34]:

p_test = test_result["hasPower"].to_numpy()
eb_low = test_result["EB_low_SOC"].to_numpy().astype(bool)
pb_low = test_result["PB_low_SOC"].to_numpy().astype(bool)

normal_soc = (~eb_low) & (~pb_low)
traction = p_test > EPS_POWER_W
regeneration = p_test < -EPS_POWER_W

detailed_regimes = {
    "normal_SOC_traction": normal_soc & traction,
    "normal_SOC_regeneration": normal_soc & regeneration,
    "EB_low_traction": eb_low & traction,
    "EB_low_regeneration": eb_low & regeneration,
    "PB_low_traction": pb_low & traction,
    "PB_low_regeneration": pb_low & regeneration,
    "traction_below_EB_limit": (
        traction
        & (p_test <= P_EB_MAX_W)
    ),
    "traction_above_EB_limit": (
        p_test > P_EB_MAX_W
    ),
    "regeneration_within_EB_limit": (
        regeneration
        & (p_test >= P_EB_MIN_W)
    ),
    "regeneration_beyond_EB_limit": (
        p_test < P_EB_MIN_W
    ),
}

rows = []

for name, mask in detailed_regimes.items():
    mask = mask & valid

    if mask.sum() == 0:
        continue

    fuzzy_values = alpha_fuzzy[mask]
    priority_values = alpha_priority[mask]

    rows.append({
        "regime": name,
        "samples": int(mask.sum()),
        "alpha_fuzzy_mean": float(
            fuzzy_values.mean()
        ),
        "alpha_priority_mean": float(
            priority_values.mean()
        ),
        "mae_alpha": float(
            np.mean(
                np.abs(
                    fuzzy_values
                    - priority_values
                )
            )
        ),
        "mean_difference": float(
            np.mean(
                fuzzy_values
                - priority_values
            )
        ),
    })

detailed_regime_df = pd.DataFrame(rows)

display(detailed_regime_df)

detailed_regime_df.to_csv(
    TABLES_DIR
    / "fuzzy_vs_eb_priority_detailed_regimes.csv",
    index=False,
)


,regime,samples,alpha_fuzzy_mean,alpha_priority_mean,mae_alpha,mean_difference
0,normal_SOC_traction,653,0.489893,0.063776,0.426117,0.426117
1,normal_SOC_regeneration,332,0.744102,0.142075,0.604877,0.602027
2,EB_low_traction,1389,0.713851,0.338117,0.563243,0.375734
3,EB_low_regeneration,692,0.735429,0.178833,0.566919,0.556595
4,PB_low_traction,139,0.409153,0.812381,0.517272,-0.403228
5,PB_low_regeneration,60,0.626863,0.244125,0.439749,0.382739
6,traction_below_EB_limit,1474,0.612406,0.156716,0.566199,0.455689
7,traction_above_EB_limit,568,0.719635,0.493468,0.397924,0.226167
8,regeneration_within_EB_limit,655,0.737387,0.000000,0.737387,0.737387
9,regeneration_beyond_EB_limit,369,0.739757,0.463202,0.298480,0.276554


In [35]:
p_test = test_result["hasPower"].to_numpy()
soc_eb_test = test_result["SOC_EB"].to_numpy()
soc_pb_test = test_result["SOC_PB"].to_numpy()

normal_below_limit = (
    valid
    & (p_test > EPS_POWER_W)
    & (p_test <= P_EB_MAX_W)
    & (soc_eb_test > SOC_LOW_THRESHOLD)
    & (soc_pb_test > SOC_LOW_THRESHOLD)
)

print("Nombre de cas :", int(normal_below_limit.sum()))

print(
    "Alpha fuzzy moyen :",
    float(alpha_fuzzy[normal_below_limit].mean()),
)

print(
    "Alpha EB-priority moyen :",
    float(alpha_priority[normal_below_limit].mean()),
)

Nombre de cas : 526
Alpha fuzzy moyen : 0.42709125475285165
Alpha EB-priority moyen : 0.0


In [36]:
normal_below_limit_indices = np.where(
    normal_below_limit
)[0]

normal_below_limit_df = test_result.loc[
    normal_below_limit
].copy()

rule_distribution = (
    normal_below_limit_df[
        "fuzzy_dominant_rule"
    ]
    .value_counts(dropna=False)
    .rename_axis("dominant_rule")
    .reset_index(name="samples")
)

rule_distribution["percent"] = (
    100.0
    * rule_distribution["samples"]
    / len(normal_below_limit_df)
)

display(rule_distribution)

,dominant_rule,samples,percent
0,DEFAULT,364,69.201521
1,R3_strong_traction,151,28.707224
2,R4_zero_demand,11,2.091255


In [38]:
rule_alpha_summary = (
    normal_below_limit_df
    .groupby(
        "fuzzy_dominant_rule",
        dropna=False,
    )
    .agg(
        samples=("alpha_ems_fuzzy_logic", "size"),
        alpha_mean=("alpha_ems_fuzzy_logic", "mean"),
        Pdem_mean=("hasPower", "mean"),
        Pdem_min=("hasPower", "min"),
        Pdem_max=("hasPower", "max"),
    )
    .reset_index()
)

display(rule_alpha_summary)

,fuzzy_dominant_rule,samples,alpha_mean,Pdem_mean,Pdem_min,Pdem_max
0,DEFAULT,364,0.30,3879.683434,168.58,7543.86
1,R3_strong_traction,151,0.75,9681.898543,7580.72,12351.93
2,R4_zero_demand,11,0.20,154.371818,103.21,189.29


In [39]:
for split_name, data in fuzzy_datasets.items():
    invalid_r5 = data[
        (data["fuzzy_dominant_rule"] == "R5_regenerative_braking")
        & (data["hasPower"] >= -EPS_POWER_W)
    ]

    print(
        split_name,
        "activations incorrectes R5 :",
        len(invalid_r5),
    )

train activations incorrectes R5 : 0
validation activations incorrectes R5 : 0
test activations incorrectes R5 : 0
